# DLE602 Week 4 Interactive Tutorial  
## Regularization, Overfitting, Underfitting and Generalisation

This notebook helps you explore how model complexity and regularization affect:

- training error,
- testing error,
- overfitting,
- underfitting,
- generalisation.

The key experiment is to adjust the **regularization parameter λ (lambda)** and observe how the model changes.

## Learning Outcomes

By the end of this notebook, you should be able to:

- Explain overfitting and underfitting using visual examples.
- Interpret training error and testing error.
- Explain why a model with perfect training performance may fail on unseen data.
- Describe how regularization helps reduce overfitting.
- Experiment with λ and observe its effect on model generalisation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from ipywidgets import interact, FloatLogSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

np.random.seed(42)
print("Libraries loaded.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Part 1: Create a Simple Dataset

We create a small dataset with a curved pattern and random noise.

A good model should learn the underlying pattern, not simply memorise every noisy point.

In [ ]:
def generate_data(n_samples=30, noise=0.25, seed=42):
    rng = np.random.default_rng(seed)
    X = np.linspace(-3, 3, n_samples)
    y_true = np.sin(X)
    y = y_true + rng.normal(0, noise, size=n_samples)
    return X.reshape(-1, 1), y.reshape(-1, 1), y_true.reshape(-1, 1)

X, y, y_true = generate_data()

plt.figure(figsize=(7,4))
plt.scatter(X, y, label="Training data")
plt.plot(X, y_true, label="True underlying pattern")
plt.title("Training Data with Noise")
plt.xlabel("Input X")
plt.ylabel("Output y")
plt.legend()
plt.grid(True)
plt.show()

## Part 2: Train/Test Split

We divide the dataset into:

- **Training data**: used to learn the model.
- **Testing data**: used to check whether the model generalises to unseen examples.

In [ ]:
def train_test_split_simple(X, y, test_ratio=0.3, seed=42):
    rng = np.random.default_rng(seed)
    indices = np.arange(len(X))
    rng.shuffle(indices)
    test_size = int(len(X) * test_ratio)
    test_idx = indices[:test_size]
    train_idx = indices[test_size:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = train_test_split_simple(X, y)

plt.figure(figsize=(7,4))
plt.scatter(X_train, y_train, label="Training data")
plt.scatter(X_test, y_test, label="Testing data")
plt.title("Training and Testing Data")
plt.xlabel("Input X")
plt.ylabel("Output y")
plt.legend()
plt.grid(True)
plt.show()

## Part 3: Polynomial Features

A low-degree polynomial may be too simple.

A high-degree polynomial may become too complex and overfit the training data.

In [ ]:
def polynomial_features(X, degree):
    return np.hstack([X ** i for i in range(degree + 1)])

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

print("Example polynomial features for X=2 and degree=3:")
print("[1, X, X^2, X^3] =", polynomial_features(np.array([[2.0]]), degree=3))

## Part 4: Model Without Regularization

First, we fit polynomial models without regularization.

Observe how model complexity changes when the polynomial degree increases.

In [ ]:
def fit_polynomial_regression(X_train, y_train, degree):
    Phi = polynomial_features(X_train, degree)
    weights = np.linalg.pinv(Phi) @ y_train
    return weights

def predict_polynomial(X, weights):
    degree = len(weights) - 1
    Phi = polynomial_features(X, degree)
    return Phi @ weights

def plot_polynomial_model(degree=1):
    weights = fit_polynomial_regression(X_train, y_train, degree)
    X_grid = np.linspace(-3.2, 3.2, 300).reshape(-1, 1)
    y_grid = predict_polynomial(X_grid, weights)
    train_error = mse(y_train, predict_polynomial(X_train, weights))
    test_error = mse(y_test, predict_polynomial(X_test, weights))

    plt.figure(figsize=(8,5))
    plt.scatter(X_train, y_train, label="Training data")
    plt.scatter(X_test, y_test, label="Testing data")
    plt.plot(X_grid, np.sin(X_grid), linestyle="--", label="True pattern")
    plt.plot(X_grid, y_grid, label=f"Polynomial degree {degree}")
    plt.title(f"Polynomial Degree = {degree}\nTraining Error = {train_error:.4f}, Testing Error = {test_error:.4f}")
    plt.xlabel("Input X")
    plt.ylabel("Output y")
    plt.legend()
    plt.grid(True)
    plt.show()

    print("Training error:", round(train_error, 4))
    print("Testing error:", round(test_error, 4))

    if train_error > 0.25 and test_error > 0.25:
        print("Interpretation: likely underfitting. Model may be too simple.")
    elif train_error < 0.05 and test_error > train_error * 3:
        print("Interpretation: likely overfitting. Model may be memorising training data.")
    else:
        print("Interpretation: model may be generalising reasonably.")

if WIDGETS_AVAILABLE:
    interact(plot_polynomial_model, degree=IntSlider(value=1, min=1, max=15, step=1))
else:
    plot_polynomial_model(degree=1)

## Recap Checkpoint 1

Discuss:

1. What happens when the degree is too low?
2. What happens when the degree is too high?
3. Why can training error be misleading?
4. Which model would you trust for unseen data?

**Key idea:** A model should not only fit the training data. It should generalise to new data.

## Part 5: Regularization Intuition

Regularization discourages unnecessary model complexity.

It adds a penalty for large weights.

`Total cost = Prediction error + Regularization penalty`

The regularization parameter **λ (lambda)** controls how strong the penalty is.

- λ = 0 means no regularization.
- Small λ means light regularization.
- Large λ means strong regularization.
- Extremely large λ may cause underfitting.

## Human Analogy

Student A memorises every past exam question.

Student B understands the concepts.

If the exam changes, Student B is more likely to perform well.

In machine learning:

- memorising training examples = overfitting,
- learning transferable patterns = generalisation,
- regularization encourages the model to avoid memorising unnecessary details.

## Part 6: Ridge Regression / L2 Regularization

We now apply L2 regularization.

L2 regularization discourages very large weights and usually produces a smoother model.

In [ ]:
def fit_ridge_regression(X_train, y_train, degree, lambda_value=0.0):
    Phi = polynomial_features(X_train, degree)
    n_features = Phi.shape[1]
    I = np.eye(n_features)
    I[0, 0] = 0
    weights = np.linalg.pinv(Phi.T @ Phi + lambda_value * I) @ Phi.T @ y_train
    return weights

def plot_regularized_model(degree=12, lambda_value=0.01):
    weights = fit_ridge_regression(X_train, y_train, degree, lambda_value=lambda_value)
    X_grid = np.linspace(-3.2, 3.2, 300).reshape(-1, 1)
    y_grid = predict_polynomial(X_grid, weights)
    train_error = mse(y_train, predict_polynomial(X_train, weights))
    test_error = mse(y_test, predict_polynomial(X_test, weights))
    weight_size = np.sum(weights[1:] ** 2)

    plt.figure(figsize=(8,5))
    plt.scatter(X_train, y_train, label="Training data")
    plt.scatter(X_test, y_test, label="Testing data")
    plt.plot(X_grid, np.sin(X_grid), linestyle="--", label="True pattern")
    plt.plot(X_grid, y_grid, label=f"Degree {degree}, λ={lambda_value:.5f}")
    plt.title(f"Regularized Model\nTraining Error = {train_error:.4f}, Testing Error = {test_error:.4f}")
    plt.xlabel("Input X")
    plt.ylabel("Output y")
    plt.legend()
    plt.grid(True)
    plt.show()

    print("λ (lambda):", lambda_value)
    print("Training error:", round(train_error, 4))
    print("Testing error:", round(test_error, 4))
    print("Weight penalty size:", round(float(weight_size), 4))

    if lambda_value == 0:
        print("Interpretation: no regularization.")
    elif lambda_value > 100:
        print("Interpretation: very strong regularization. Risk of underfitting.")
    elif test_error < train_error * 3:
        print("Interpretation: regularization may be improving generalisation.")
    else:
        print("Interpretation: inspect the plot and errors carefully.")

if WIDGETS_AVAILABLE:
    interact(
        plot_regularized_model,
        degree=IntSlider(value=12, min=1, max=20, step=1, description="Degree"),
        lambda_value=FloatLogSlider(value=0.01, base=10, min=-5, max=4, step=0.25, description="λ")
    )
else:
    plot_regularized_model(degree=12, lambda_value=0.01)

## Recap Checkpoint 2

Move λ from very small to very large.

Observe:

1. What happens to the curve?
2. What happens to training error?
3. What happens to testing error?
4. At what point does the model become too simple?
5. Is the lowest training error always the best outcome?

**Key idea:** Regularization may slightly increase training error but improve testing performance.

## Part 7: Comparing Different λ Values

In [ ]:
def compare_lambdas(degree=12):
    lambda_values = [0, 0.0001, 0.01, 1, 100]
    X_grid = np.linspace(-3.2, 3.2, 300).reshape(-1, 1)
    plt.figure(figsize=(9,5))
    plt.scatter(X_train, y_train, label="Training data")
    plt.scatter(X_test, y_test, label="Testing data")
    plt.plot(X_grid, np.sin(X_grid), linestyle="--", label="True pattern")

    results = []
    for lam in lambda_values:
        weights = fit_ridge_regression(X_train, y_train, degree, lambda_value=lam)
        y_grid = predict_polynomial(X_grid, weights)
        train_error = mse(y_train, predict_polynomial(X_train, weights))
        test_error = mse(y_test, predict_polynomial(X_test, weights))
        results.append((lam, train_error, test_error))
        plt.plot(X_grid, y_grid, label=f"λ={lam}")

    plt.title(f"Effect of λ on Model Shape (Degree {degree})")
    plt.xlabel("Input X")
    plt.ylabel("Output y")
    plt.legend()
    plt.grid(True)
    plt.show()

    print("λ comparison")
    print("------------")
    for lam, tr, te in results:
        print(f"λ={lam:<8} | Training error={tr:.4f} | Testing error={te:.4f}")

compare_lambdas(degree=12)

## Part 8: Training Error vs Testing Error

This section plots training and testing error as λ changes.

In [ ]:
def plot_error_vs_lambda(degree=12):
    lambda_values = np.logspace(-6, 4, 80)
    train_errors = []
    test_errors = []

    for lam in lambda_values:
        weights = fit_ridge_regression(X_train, y_train, degree, lambda_value=lam)
        train_errors.append(mse(y_train, predict_polynomial(X_train, weights)))
        test_errors.append(mse(y_test, predict_polynomial(X_test, weights)))

    best_index = int(np.argmin(test_errors))
    best_lambda = lambda_values[best_index]

    plt.figure(figsize=(8,5))
    plt.semilogx(lambda_values, train_errors, label="Training error")
    plt.semilogx(lambda_values, test_errors, label="Testing error")
    plt.axvline(best_lambda, linestyle="--", label=f"Best test λ ≈ {best_lambda:.5f}")
    plt.title("Training Error vs Testing Error as λ Changes")
    plt.xlabel("λ (regularization strength)")
    plt.ylabel("Mean Squared Error")
    plt.legend()
    plt.grid(True)
    plt.show()

    print("Best λ based on testing error:", best_lambda)
    print("Minimum testing error:", min(test_errors))

plot_error_vs_lambda(degree=12)

## Discussion

Notice:

- Training error often increases as λ increases.
- Testing error may decrease first because the model generalises better.
- If λ becomes too large, both training and testing performance may become poor.
- This is underfitting caused by excessive regularization.

Regularization is therefore a balance.

## Part 9: GenAI Critique Activity

Ask ChatGPT or another GenAI tool:

> Explain regularization using the example of a student preparing for an exam.

Then critique the response:

1. Does it explain overfitting clearly?
2. Does it explain generalisation?
3. Does it explain λ?
4. Does it mention the risk of too much regularization?
5. What would you improve?

## Part 10: Assessment Connection

Think about your Assessment 1 or later project work.

Answer:

1. Where could overfitting occur in your model?
2. What would your training data be?
3. What would your test data be?
4. How would you know if your model generalises?
5. Which regularization method might be useful?

## Final Reflection

Complete the following:

1. Overfitting means...
2. Underfitting means...
3. Generalisation means...
4. Regularization helps because...
5. If λ is too large...
6. A good model should perform well on...